# Build an adjusted price series

## Goal

Produce a table of raw closes, factors, the anchor and adjusted closes; do not calculate strategy returns.

This notebook uses synthetic teaching data, not a paper replication or production observations.

## Setup

Use a Python 3.10+ kernel and run all cells in order. Computation uses only the standard library, without keys, networking or extra data files. Open in an existing Jupyter environment.

Embedded inputs match inputs.json in the same download directory. Edit args in the next cell to experiment; preserve explicit times and units.

In [ ]:
import json

# Synthetic inputs; no credentials or network access.
bundle = json.loads("{\"version\":1,\"tutorial\":\"adjusted-price-series\",\"identity\":\"synthetic\",\"args\":[[{\"security\":\"DEMO\",\"date\":\"2025-01-02\",\"close\":100,\"factor\":1},{\"security\":\"DEMO\",\"date\":\"2025-01-03\",\"close\":50,\"factor\":2},{\"security\":\"DEMO\",\"date\":\"2025-01-06\",\"close\":51,\"factor\":2}],\"2025-01-06\"],\"expected\":[{\"security\":\"DEMO\",\"date\":\"2025-01-02\",\"close\":100,\"factor\":1,\"anchorDate\":\"2025-01-06\",\"adjustedClose\":50},{\"security\":\"DEMO\",\"date\":\"2025-01-03\",\"close\":50,\"factor\":2,\"anchorDate\":\"2025-01-06\",\"adjustedClose\":50},{\"security\":\"DEMO\",\"date\":\"2025-01-06\",\"close\":51,\"factor\":2,\"anchorDate\":\"2025-01-06\",\"adjustedClose\":51}]}")
args = bundle["args"]
expected = bundle["expected"]
print(json.dumps(args, ensure_ascii=False, indent=2))

## Steps

### 1. Freeze the price definition

Confirm that inputs are unadjusted, not previously adjusted. Tushare's daily specification distinguishes close from the ex-rights reference pre_close; do not mix them. Retain source fields, currency and version, and check uniqueness by security and trading date.

### 2. Join factors on the same date

Join on the exact security and trading date. Missing factors, duplicate keys and non-positive prices or factors must stop the calculation, not default to one. No trade during a suspension does not mean a zero price; retain a calendar and missingness reasons separately.

### 3. Declare the normalization anchor

For multiplicative adjustment factors, the example uses adjusted_close = close × factor / anchor_factor. With the final date as anchor, its adjusted close equals its raw close. Changing the anchor rescales the series, so retain the actual date rather than only a label such as forward-adjusted.

### 4. Check jumps and units

In the synthetic two-for-one split, price changes from 100 to 50 and the factor from 1 to 2; both adjusted observations are 50. This is arithmetic, not an observed corporate action. Do not apply the price factor blindly to volume or traded value.

### Method and assumptions

- Applying adjustment twice creates false changes; reject already adjusted inputs.
- The latest factor version is not proof of historical availability; freeze factor snapshots.
- Adjusted prices are not automatically total returns with taxes, fees and dividend reinvestment.

In [ ]:
def adjust_prices(rows, anchor_date):
    """Normalize one synthetic price series to the selected factor anchor."""
    from datetime import date
    import math

    if not rows:
        raise ValueError("empty_input")
    seen = set()
    security = rows[0].get("security")
    for row in rows:
        if not security or row.get("security") != security:
            raise ValueError("one_security_required")
        day = row.get("date", "")
        try:
            valid_date = date.fromisoformat(day).isoformat() == day
        except (ValueError, TypeError):
            valid_date = False
        if not valid_date or day in seen:
            raise ValueError("invalid_or_duplicate_date")
        for field in ("close", "factor"):
            value = row.get(field)
            if type(value) not in (int, float) or not math.isfinite(value) or value <= 0:
                raise ValueError("invalid_price_or_factor")
        seen.add(day)
    anchor = next((row for row in rows if row["date"] == anchor_date), None)
    if anchor is None:
        raise ValueError("missing_anchor")
    return [dict(row, anchorDate=anchor_date,
                 adjustedClose=round(row["close"] * row["factor"] / anchor["factor"], 6))
            for row in sorted(rows, key=lambda item: item["date"])]


### Run the sample

Three rows: adjustedClose is 50, 50, 51; raw close remains 100, 50, 51. The anchor price is unchanged and inputs are not overwritten.

In [ ]:
result = adjust_prices(*args)
print(json.dumps(result, ensure_ascii=False, indent=2))

## Checks

Compare every row with the browser example's expected output. After editing inputs, a failed assertion may be expected: explain the difference before changing the check.

In [ ]:
assert result == expected, "Output differs from the reference synthetic example"
assert bundle["identity"] == "synthetic"
print("Passed: output matches the synthetic browser example.")

## Next steps

Before real data, confirm grants, fields, schema_major, windows and provenance using authenticated GET /v1/catalog, then map the actual contract. Candidate IDs below do not guarantee availability or historical completeness. API as_of is not a historical filing-version guarantee. Validate again after substituting real inputs; the synthetic pass does not transfer.

- `cn.equity.daily`
- `cn.dataset.adj_factor`

### References

- [Tushare: adjustment-factor fields](https://tushare.pro/document/2?doc_id=28)
- [Tushare: daily-price conventions](https://tushare.pro/document/2?doc_id=27)

[Back to tutorial](https://tradingdatas.com/recipes/adjusted-price-series/)